In [ ]:
# !pip install -U pandas
# !pip install plotly
# !pip install nbformat

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
import plotly.express as px
import os


# ایجاد پوشه خروجی
output_dir = "reports/torus"
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

def read_protein_data(file_path):
    """Read protein data from TSV file"""
    raw_data = pd.read_csv(file_path, delimiter="\t", header=None)
    raw_data.columns = ["name", "phi", "psi", "subtype"]
    return raw_data

def extract_protein_name(name: str) -> str:
    """Extract protein name from complex name field"""
    p_name = str(name).split(":")[0]
    return p_name.upper()

# خواندن فایل داده‌ها
file_path = "fff/data/raw_data/torus/protein.tsv"
protein_data = read_protein_data(file_path)

# استخراج نام پروتئین‌ها
protein_data['protein_name'] = protein_data['name'].apply(extract_protein_name)

# نمایش اطلاعات اولیه
print(f"Total data points: {len(protein_data)}")
print(f"Unique proteins: {protein_data['protein_name'].nunique()}")
print(f"Protein names: {protein_data['protein_name'].unique()[:10]}")  # 10 پروتئین اول

# انتخاب زیرمجموعه برای نمایش (اگر داده‌ها زیاد هستند)
sample_data = protein_data  # یا protein_data.sample(1000) برای نمونه‌گیری
# sample_data = protein_data.sample(10000)


In [ ]:
def plot_protein_torus_matplotlib(data, max_proteins=100, title="Protein Data on Torus"):
    """Plot protein data on torus with flatter shape"""
    
    # محدود کردن تعداد پروتئین‌ها
    unique_proteins = data['protein_name'].unique()
    if len(unique_proteins) > max_proteins:
        print(f"Too many proteins ({len(unique_proteins)}). Sampling {max_proteins} proteins...")
        protein_counts = data['protein_name'].value_counts()
        selected_proteins = protein_counts.head(max_proteins).index
        data = data[data['protein_name'].isin(selected_proteins)]
        unique_proteins = selected_proteins
    
    phi_degrees = data['phi'].values
    psi_degrees = data['psi'].values
    protein_names = data['protein_name'].values
    
    # تبدیل به رادیان
    phi_rad = np.deg2rad(phi_degrees)
    psi_rad = np.deg2rad(psi_degrees)
    
    # پارامترهای توروس - دونات کشیده‌تر در صفحه XY
    R, r = 4.0, 0.8  # افزایش شعاع اصلی و کاهش شعاع فرعی برای کشیدگی بیشتر
    
    # تبدیل به مختصات 3 بعدی
    x_data = (R + r * np.cos(phi_rad)) * np.cos(psi_rad)
    y_data = (R + r * np.cos(phi_rad)) * np.sin(psi_rad)
    z_data = r * np.sin(phi_rad)
    
    # ایجاد نقشه رنگ
    unique_proteins = list(set(protein_names))
    color_map = plt.cm.tab20(np.linspace(0, 1, len(unique_proteins)))
    protein_to_color = {prot: color_map[i] for i, prot in enumerate(unique_proteins)}
    
    # ایجاد figure
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # رسم سطح توروس با جزئیات کمتر برای عملکرد بهتر
    u = np.linspace(0, 2*np.pi, 40)
    v = np.linspace(0, 2*np.pi, 40)
    U, V = np.meshgrid(u, v)
    
    X = (R + r * np.cos(U)) * np.cos(V)
    Y = (R + r * np.cos(U)) * np.sin(V)
    Z = r * np.sin(U)
    
    ax.plot_wireframe(X, Y, Z, alpha=0.15, color='gray', rstride=3, cstride=3, linewidth=0.5)
    
    # رسم داده‌ها
    colors = [protein_to_color[prot] for prot in protein_names]
    scatter = ax.scatter(x_data, y_data, z_data,
                       c=colors, s=12, alpha=0.6,
                       edgecolors='black', linewidth=0.1)
    
    # تنظیمات نمودار
    ax.set_xlabel('X', fontsize=12)
    ax.set_ylabel('Y', fontsize=12)
    ax.set_zlabel('Z', fontsize=12)
    ax.set_title(f"{title}\n({len(unique_proteins)} proteins, {len(data)} data points)", 
                fontsize=14, pad=20)
    
    # تنظیم نسبت ابعاد برای کشیدگی بیشتر در XY
    ax.set_box_aspect([2, 2, 1])  # کشیده‌تر در صفحه XY
    
    # تنظیم محدوده axes برای نمایش بهتر
    max_range = R + r + 1
    ax.set_xlim(-max_range, max_range)
    ax.set_ylim(-max_range, max_range)
    ax.set_zlim(-r-0.5, r+0.5)
    
    ax.grid(True, alpha=0.2)
    
    plt.tight_layout()
    return fig, ax

def plot_protein_torus_plotly(data, max_proteins=50, title="Protein Torsion Angles on Torus"):
    """Create interactive torus plot with flatter shape"""
    
    # محدود کردن تعداد پروتئین‌ها
    unique_proteins = data['protein_name'].unique()
    if len(unique_proteins) > max_proteins:
        protein_counts = data['protein_name'].value_counts()
        selected_proteins = protein_counts.head(max_proteins).index
        data = data[data['protein_name'].isin(selected_proteins)]
        unique_proteins = selected_proteins
    
    phi_degrees = data['phi'].values
    psi_degrees = data['psi'].values
    protein_names = data['protein_name'].values
    full_names = data['name'].values
    
    # تبدیل به رادیان و مختصات 3 بعدی با شکل کشیده
    phi_rad = np.deg2rad(phi_degrees)
    psi_rad = np.deg2rad(psi_degrees)
    R, r = 4.0, 0.8  # کشیده‌تر در صفحه XY
    
    x_data = (R + r * np.cos(phi_rad)) * np.cos(psi_rad)
    y_data = (R + r * np.cos(phi_rad)) * np.sin(psi_rad)
    z_data = r * np.sin(phi_rad)
    
    # ایجاد متن hover
    hover_text = [
        f"Protein: {prot}<br>Residue: {name.split(':')[-1]}<br>φ: {phi:.1f}°<br>ψ: {psi:.1f}°"
        for prot, name, phi, psi in zip(protein_names, full_names, phi_degrees, psi_degrees)
    ]
    
    # ایجاد figure
    fig = go.Figure()
    
    # اضافه کردن داده‌ها
    fig.add_trace(go.Scatter3d(
        x=x_data,
        y=y_data, 
        z=z_data,
        mode='markers',
        marker=dict(
            size=3,
            opacity=0.7,
            color=phi_degrees,
            colorscale='Viridis',
            colorbar=dict(title="Phi (°)", x=1.1),
            line=dict(width=0.3, color='black')
        ),
        text=hover_text,
        hoverinfo='text',
        name='Protein Data',
        showlegend=False
    ))
    
    # اضافه کردن سطح توروس کشیده
    u = np.linspace(0, 2*np.pi, 25)
    v = np.linspace(0, 2*np.pi, 25)
    U, V = np.meshgrid(u, v)
    
    X = (R + r * np.cos(U)) * np.cos(V)
    Y = (R + r * np.cos(U)) * np.sin(V)
    Z = r * np.sin(U)
    
    fig.add_trace(go.Surface(
        x=X, y=Y, z=Z,
        opacity=0.1,
        colorscale='Greys',
        showscale=False,
        hoverinfo='skip',
        name='Torus Surface',
        showlegend=False
    ))
    
    # تنظیمات layout برای شکل کشیده
    fig.update_layout(
        title=dict(
            text=f"{title}<br><sup>{len(unique_proteins)} proteins, {len(data)} data points</sup>",
            x=0.5,
            xanchor='center',
            y=0.95
        ),
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y', 
            zaxis_title='Z',
            aspectratio=dict(x=2, y=2, z=1),  # کشیدگی در XY
            camera=dict(
                eye=dict(x=1.8, y=1.8, z=0.8),  # زاویه دید برای نمایش کشیدگی
                up=dict(x=0, y=0, z=1)
            ),
            xaxis=dict(range=[-R-r-1, R+r+1]),
            yaxis=dict(range=[-R-r-1, R+r+1]),
            zaxis=dict(range=[-r-1, r+1])
        ),
        width=1000,
        height=700,
        showlegend=False,
        margin=dict(l=0, r=0, b=0, t=100)
    )
    
    return fig

In [ ]:
def plot_proteins_separately(data, n_proteins=9, samples_per_protein=200):
    """Plot multiple proteins in separate subplots with flatter torus"""
    
    top_proteins = data['protein_name'].value_counts().head(n_proteins).index
    
    n_cols = min(3, n_proteins)
    n_rows = (n_proteins + n_cols - 1) // n_cols
    
    fig = plt.figure(figsize=(6*n_cols, 5*n_rows))
    
    R, r = 4.0, 0.8  # شکل کشیده
    
    for i, protein in enumerate(top_proteins):
        if i >= n_proteins:
            break
            
        ax = fig.add_subplot(n_rows, n_cols, i+1, projection='3d')
        
        # داده‌های این پروتئین
        protein_data = data[data['protein_name'] == protein]
        if len(protein_data) > samples_per_protein:
            protein_data = protein_data.sample(samples_per_protein)
        
        phi_degrees = protein_data['phi'].values
        psi_degrees = protein_data['psi'].values
        
        # تبدیل به مختصات
        phi_rad = np.deg2rad(phi_degrees)
        psi_rad = np.deg2rad(psi_degrees)
        
        x_data = (R + r * np.cos(phi_rad)) * np.cos(psi_rad)
        y_data = (R + r * np.cos(phi_rad)) * np.sin(psi_rad)
        z_data = r * np.sin(phi_rad)
        
        # رسم
        ax.scatter(x_data, y_data, z_data, s=25, alpha=0.8, color=f'C{i}', edgecolors='black', linewidth=0.3)
        
        # رسم توروس
        u = np.linspace(0, 2*np.pi, 20)
        v = np.linspace(0, 2*np.pi, 20)
        U, V = np.meshgrid(u, v)
        
        X = (R + r * np.cos(U)) * np.cos(V)
        Y = (R + r * np.cos(U)) * np.sin(V)
        Z = r * np.sin(U)
        
        ax.plot_wireframe(X, Y, Z, alpha=0.1, color='gray', linewidth=0.5)
        
        ax.set_title(f'{protein}\n({len(protein_data)} points)', fontsize=11)
        ax.set_box_aspect([2, 2, 1])  # کشیده در XY
        
        # تنظیم محدوده‌ها
        max_range = R + r + 1
        ax.set_xlim(-max_range, max_range)
        ax.set_ylim(-max_range, max_range)
        ax.set_zlim(-r-0.5, r+0.5)
        
        ax.grid(True, alpha=0.2)
    
    plt.tight_layout()
    return fig

In [ ]:
# خواندن داده‌ها
file_path = "fff/data/raw_data/torus/protein.tsv"
protein_data = pd.read_csv(file_path, delimiter="\t", header=None)
protein_data.columns = ["name", "phi", "psi", "subtype"]
protein_data['protein_name'] = protein_data['name'].apply(lambda x: str(x).split(":")[0].upper())

print(f"Total data points: {len(protein_data)}")
print(f"Unique proteins: {protein_data['protein_name'].nunique()}")

# ایجاد نمودارها و ذخیره در پوشه reports/torus
print("\nCreating plots...")

# 1. نمودار با 100 پروتئین
fig1, ax1 = plot_protein_torus_matplotlib(protein_data, max_proteins=494)
plt.savefig(f'{output_dir}/protein_torus_100_proteins.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{output_dir}/protein_torus_100_proteins.pdf', bbox_inches='tight')
plt.show()

# 2. نمودار تعاملی با 50 پروتئین
fig2 = plot_protein_torus_plotly(protein_data, max_proteins=494)
fig2.write_html(f"{output_dir}/protein_torus_50_proteins_interactive.html")
fig2.show()

# 3. نمودارهای جداگانه
fig3 = plot_proteins_separately(protein_data, n_proteins=9, samples_per_protein=200)
plt.savefig(f'{output_dir}/protein_subplots_9_proteins.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{output_dir}/protein_subplots_9_proteins.pdf', bbox_inches='tight')
plt.show()

# 4. نمودار ساده‌تر با 20 پروتئین
fig4, ax4 = plot_protein_torus_matplotlib(protein_data, max_proteins=20, 
                                        title="Top 20 Proteins on Torus")
plt.savefig(f'{output_dir}/protein_torus_20_proteins.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{output_dir}/protein_torus_20_proteins.pdf', bbox_inches='tight')
plt.show()

print(f"\nAll plots saved to: {output_dir}/")
print("✓ protein_torus_100_proteins.png/.pdf")
print("✓ protein_torus_50_proteins_interactive.html") 
print("✓ protein_subplots_9_proteins.png/.pdf")
print("✓ protein_torus_20_proteins.png/.pdf")

In [ ]:
# تحلیل آماری داده‌ها
print("\n=== Statistical Analysis ===")

# تعداد داده‌ها به ازای هر پروتئین
protein_counts = protein_data['protein_name'].value_counts()
print(f"\nData points per protein:")
print(protein_counts.head(10))  # 10 پروتئین اول

# محدوده زوایا برای هر پروتئین
angle_stats = protein_data.groupby('protein_name').agg({
    'phi': ['min', 'max', 'mean', 'std'],
    'psi': ['min', 'max', 'mean', 'std']
}).round(1)

print(f"\nAngle statistics per protein:")
print(angle_stats.head())

# رسم هیستوگرام توزیع داده‌ها بین پروتئین‌ها
plt.figure(figsize=(12, 6))
protein_counts.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Number of Data Points per Protein')
plt.xlabel('Protein Name')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()